In [ ]:
# face crops for gallery using RetinaFace

import os
import cv2
import pandas as pd

from tqdm import tqdm
from retinaface import RetinaFace

INPUT_ROOT = "natural_test/gallery"
OUTPUT_ROOT = "natural_test/gallery_cropped"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

failed_images = []
metadata_rows = []

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)

    if not os.path.isdir(input_dir):
        continue

    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        try:

            img = cv2.imread(image_path)

            if img is None:
                failed_images.append(image_path)
                continue

            h, w = img.shape[:2]

            # RetinaFace performs better on moderate resolutions
            max_size = 1000

            if max(h, w) > max_size:

                scale = max_size / max(h, w)

                new_w = int(w * scale)
                new_h = int(h * scale)

                img_detect = cv2.resize(img, (new_w, new_h))

            else:

                img_detect = img.copy()
                scale = 1.0

            faces = RetinaFace.detect_faces(img_detect)

            if not isinstance(faces, dict):

                print(f"NO FACE: {image_file}")

                failed_images.append(image_path)

                continue

            largest_area = -1
            best_box = None
            best_score = None

            for face_id in faces:

                x1, y1, x2, y2 = faces[face_id]["facial_area"]

                area = (x2 - x1) * (y2 - y1)

                if area > largest_area:

                    largest_area = area

                    best_box = (x1, y1, x2, y2)

                    best_score = float(faces[face_id]["score"])

            if best_box is None:

                failed_images.append(image_path)

                continue

            x1, y1, x2, y2 = best_box

            # map coordinates back to original image

            x1 = int(x1 / scale)
            y1 = int(y1 / scale)

            x2 = int(x2 / scale)
            y2 = int(y2 / scale)

            # padding around face

            pad = 20

            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)

            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            crop = img[y1:y2, x1:x2]

            if crop.size == 0:

                failed_images.append(image_path)

                continue

            crop = cv2.resize(crop, (112, 112))

            save_path = os.path.join(output_dir, image_file)

            cv2.imwrite(save_path, crop)

            metadata_rows.append(
                {
                    "identity": identity,
                    "image_name": image_file,
                    "det_score": best_score,
                }
            )

            print(f"{identity} | {image_file} | Detection Score = {best_score:.6f}")

        except Exception as e:

            print(f"ERROR: {image_path}")
            print(e)

            failed_images.append(image_path)

# save failed detections

pd.DataFrame({"failed_image": failed_images}).to_csv(
    "gallery_failed_detections.csv", index=False
)

# save detection scores

pd.DataFrame(metadata_rows).to_csv("gallery_detection_scores.csv", index=False)

print("\nGallery Cropping Done")

print("Successful Crops:", len(metadata_rows))
print("Failed Images:", len(failed_images))

In [ ]:
# face crop for probes using RetinaFace

import os
import cv2

from tqdm import tqdm
from retinaface import RetinaFace

INPUT_ROOT = "natural_test/probes"
OUTPUT_ROOT = "natural_test/probes_cropped"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)

    if not os.path.isdir(input_dir):
        continue

    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        try:

            img = cv2.imread(image_path)

            if img is None:
                continue

            faces = RetinaFace.detect_faces(image_path)

            if not isinstance(faces, dict):
                print(f"No face detected: {image_file}")
                continue

            largest_area = -1
            best_box = None
            best_score = None

            for face_id in faces:

                x1, y1, x2, y2 = faces[face_id]["facial_area"]

                area = (x2 - x1) * (y2 - y1)

                if area > largest_area:

                    largest_area = area
                    best_box = (x1, y1, x2, y2)
                    best_score = faces[face_id]["score"]

            if best_box is None:
                continue

            x1, y1, x2, y2 = best_box

            h, w = img.shape[:2]

            x1 = max(0, x1)
            y1 = max(0, y1)

            x2 = min(w, x2)
            y2 = min(h, y2)

            crop = img[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            crop = cv2.resize(crop, (112, 112))

            save_path = os.path.join(output_dir, image_file)

            cv2.imwrite(save_path, crop)

            print(f"{identity} | {image_file} | Detection Score = {best_score:.6f}")

        except Exception as e:

            print(f"\nERROR: {image_path}")
            print(e)

print("Probe Cropping Done")

In [ ]:
for identity in os.listdir("natural_test/gallery"):

    src = os.path.join("natural_test/gallery", identity)
    dst = os.path.join("natural_test/gallery_cropped", identity)

    if not os.path.isdir(src):
        continue

    original = len(os.listdir(src))

    cropped = 0

    if os.path.exists(dst):
        cropped = len(os.listdir(dst))

    print(identity, original, cropped)

In [ ]:
for identity in os.listdir("natural_test/gallery"):

    src = os.path.join("natural_test/gallery", identity)
    dst = os.path.join("natural_test/gallery_cropped", identity)

    if not os.path.isdir(src):
        continue

    cropped_files = set()

    if os.path.exists(dst):
        cropped_files = set(os.listdir(dst))

    for img in os.listdir(src):

        if img not in cropped_files:

            print(identity, img)

In [ ]:
# face crops for unknown images
import os
import cv2
from tqdm import tqdm
from insightface.app import FaceAnalysis

INPUT_ROOT = "natural_test/unknown"
OUTPUT_ROOT = "natural_test/unknown_cropped"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)

    # skip .DS_Store and other non-folders
    if not os.path.isdir(input_dir):
        continue

    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        img = cv2.imread(image_path)

        if img is None:
            continue

        faces = app.get(img)

        if len(faces) == 0:
            continue

        best_face = max(faces, key=lambda x: x.det_score)

        bbox = best_face.bbox.astype(int)

        x1, y1, x2, y2 = bbox

        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        save_path = os.path.join(output_dir, image_file)

        cv2.imwrite(save_path, crop)

print("Unknown Cropping Done")

In [ ]:
# generating arcface embeddings
import os
import cv2
import numpy as np

from tqdm import tqdm
from insightface.app import FaceAnalysis

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

rec_model = app.models["recognition"]


def generate_embeddings(INPUT_ROOT, OUTPUT_ROOT):

    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    count = 0

    for identity in tqdm(os.listdir(INPUT_ROOT)):

        input_dir = os.path.join(INPUT_ROOT, identity)
        output_dir = os.path.join(OUTPUT_ROOT, identity)

        os.makedirs(output_dir, exist_ok=True)

        for image_file in os.listdir(input_dir):

            image_path = os.path.join(input_dir, image_file)

            img = cv2.imread(image_path)

            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (112, 112))

            embedding = rec_model.get_feat(img).flatten().astype(np.float32)

            save_path = os.path.join(
                output_dir, os.path.splitext(image_file)[0] + ".npy"
            )

            np.save(save_path, embedding)

            count += 1

    print("Saved:", count)


generate_embeddings("natural_test/gallery_cropped", "natural_test/gallery_embeddings")

generate_embeddings("natural_test/probes_cropped", "natural_test/probe_embeddings")

generate_embeddings("natural_test/unknown_cropped", "natural_test/unknown_embeddings")

In [32]:
# creating dataset features
import os
import torch
import pyiqa
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

musiq_metric = pyiqa.create_metric("musiq", device=device)


def get_quality_score(image_path):

    score = musiq_metric(image_path)

    return float(score.cpu().item())

Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth


In [33]:
def load_gallery_db(gallery_root):

    gallery_db = {}

    for identity in os.listdir(gallery_root):

        identity_dir = os.path.join(gallery_root, identity)

        # skip .DS_Store
        if not os.path.isdir(identity_dir):
            continue

        embeddings = []

        for file in os.listdir(identity_dir):

            if not file.endswith(".npy"):
                continue

            emb = np.load(os.path.join(identity_dir, file))

            embeddings.append(emb)

        # skip empty identities
        if len(embeddings) == 0:
            continue

        gallery_db[identity] = embeddings

    return gallery_db

In [34]:
# gallery search
def search_gallery(probe_embedding, gallery_db):

    identity_scores = {}

    for identity in gallery_db:

        sims = []

        for gallery_emb in gallery_db[identity]:

            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]

            sims.append(sim)

        identity_scores[identity] = max(sims)

    sorted_scores = sorted(identity_scores.items(), key=lambda x: x[1], reverse=True)

    predicted_identity = sorted_scores[0][0]

    best_similarity = sorted_scores[0][1]

    second_similarity = sorted_scores[1][1]

    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)

In [ ]:
for identity in os.listdir("natural_test/gallery_embeddings"):

    identity_dir = os.path.join("natural_test/gallery_embeddings", identity)

    if not os.path.isdir(identity_dir):
        continue

    n = len(os.listdir(identity_dir))

    print(identity, n)

In [ ]:
# known natural probes dataset
gallery_db = load_gallery_db("natural_test/gallery_embeddings")

rows = []

probe_emb_root = "natural_test/probe_embeddings"
probe_img_root = "natural_test/probes_cropped"

for identity in tqdm(os.listdir(probe_emb_root)):

    emb_dir = os.path.join(probe_emb_root, identity)

    # skip .DS_Store
    if not os.path.isdir(emb_dir):
        continue

    img_dir = os.path.join(probe_img_root, identity)

    for emb_file in os.listdir(emb_dir):

        if not emb_file.endswith(".npy"):
            continue

        probe_embedding = np.load(os.path.join(emb_dir, emb_file))

        base_name = os.path.splitext(emb_file)[0]

        image_path = None

        for ext in [".jpg", ".jpeg", ".png"]:

            candidate = os.path.join(img_dir, base_name + ext)

            if os.path.exists(candidate):

                image_path = candidate
                break

        # image not found
        if image_path is None:
            continue

        # MUSIQ quality score
        quality_score = get_quality_score(image_path)

        # gallery search
        predicted_identity, best_similarity, second_similarity, margin = search_gallery(
            probe_embedding, gallery_db
        )

        # ground truth
        label = int(predicted_identity == identity)

        rows.append(
            {
                "quality_score": quality_score,
                "best_similarity": best_similarity,
                "margin": margin,
                "label": label,
                "true_identity": identity,
                "predicted_identity": predicted_identity,
            }
        )

natural_df = pd.DataFrame(rows)

natural_df.to_csv("natural_probe_dataset.csv", index=False)

print("Saved natural_probe_dataset.csv")
print("Samples:", len(natural_df))

print("\nLabel Distribution")
print(natural_df["label"].value_counts())

In [37]:
# runing saved svm model
import joblib

svm = joblib.load("svm_rejection_model.pkl")

scaler = joblib.load("svm_scaler.pkl")

df = pd.read_csv("natural_probe_dataset.csv")

X = df[["quality_score", "best_similarity", "margin"]]

X_scaled = scaler.transform(X)

df["svm_decision"] = svm.predict(X_scaled)

df["confidence_score"] = svm.predict_proba(X_scaled)[:, 1]

In [ ]:
# svm metrics
from sklearn.metrics import *

y_true = df["label"]
y_pred = df["svm_decision"]

cm = confusion_matrix(y_true, y_pred)

print(cm)

print(classification_report(y_true, y_pred))

TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)
FRR = FN / (TP + FN)

TRR = TN / (TN + FP)
FAR = FP / (TN + FP)
svm_accuracy = accuracy_score(y_true, y_pred)

svm_TAR = TAR
svm_FRR = FRR
svm_TRR = TRR
svm_FAR = FAR

print("Accuracy:", svm_accuracy)

print("TAR:", svm_TAR)
print("FRR:", svm_FRR)

print("TRR:", svm_TRR)
print("FAR:", svm_FAR)

In [ ]:
# fixed threshold method
THRESHOLD = 0.60

threshold_pred = (df["best_similarity"] >= THRESHOLD).astype(int)

cm = confusion_matrix(y_true, threshold_pred)

print(cm)

print(classification_report(y_true, threshold_pred))

TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)
FRR = FN / (TP + FN)

TRR = TN / (TN + FP)
FAR = FP / (TN + FP)
threshold_accuracy = accuracy_score(y_true, threshold_pred)

threshold_TAR = TAR
threshold_FRR = FRR
threshold_TRR = TRR
threshold_FAR = FAR

print("Accuracy:", threshold_accuracy)

print("TAR:", threshold_TAR)
print("FRR:", threshold_FRR)

print("TRR:", threshold_TRR)
print("FAR:", threshold_FAR)

In [41]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# SVM metrics
svm_precision = precision_score(y_true, y_pred)
svm_recall = recall_score(y_true, y_pred)
svm_f1 = f1_score(y_true, y_pred)

# Threshold metrics
threshold_precision = precision_score(y_true, threshold_pred)
threshold_recall = recall_score(y_true, threshold_pred)
threshold_f1 = f1_score(y_true, threshold_pred)

In [ ]:
# comparison table
comparison = pd.DataFrame(
    {
        "Method": ["Fixed Threshold", "SVM"],
        "Accuracy": [threshold_accuracy, svm_accuracy],
        "Precision": [threshold_precision, svm_precision],
        "Recall": [threshold_recall, svm_recall],
        "F1": [threshold_f1, svm_f1],
        "TAR": [threshold_TAR, svm_TAR],
        "FRR": [threshold_FRR, svm_FRR],
        "TRR": [threshold_TRR, svm_TRR],
        "FAR": [threshold_FAR, svm_FAR],
    }
)

print(comparison)

In [ ]:
import pandas as pd

train_df = pd.read_csv("train_svm_extended_all.csv")

print("\nBEST SIMILARITY")
print(train_df["best_similarity"].describe())

print("\nMARGIN")
print(train_df["margin"].describe())

print("\nQUALITY SCORE")
print(train_df["quality_score"].describe())

if "det_score" in train_df.columns:
    print("\nDETECTION SCORE")
    print(train_df["det_score"].describe())

In [ ]:
print("\n========== ACCEPTS (label=1) ==========")

print("\nBest Similarity")
print(train_df[train_df["label"] == 1]["best_similarity"].describe())

print("\nMargin")
print(train_df[train_df["label"] == 1]["margin"].describe())

print("\nQuality")
print(train_df[train_df["label"] == 1]["quality_score"].describe())


print("\n========== REJECTS (label=0) ==========")

print("\nBest Similarity")
print(train_df[train_df["label"] == 0]["best_similarity"].describe())

print("\nMargin")
print(train_df[train_df["label"] == 0]["margin"].describe())

print("\nQuality")
print(train_df[train_df["label"] == 0]["quality_score"].describe())

In [ ]:
natural_df = pd.read_csv("natural_probe_dataset.csv")

print("\nBEST SIMILARITY")
print(natural_df["best_similarity"].describe())

print("\nMARGIN")
print(natural_df["margin"].describe())

print("\nQUALITY SCORE")
print(natural_df["quality_score"].describe())

In [ ]:
gallery_counts = []

for identity in os.listdir("natural_test/gallery_embeddings"):

    path = os.path.join("natural_test/gallery_embeddings", identity)

    if os.path.isdir(path):

        gallery_counts.append(len(os.listdir(path)))

print("Min:", min(gallery_counts))
print("Max:", max(gallery_counts))
print("Mean:", sum(gallery_counts) / len(gallery_counts))

In [ ]:
accepts = natural_df[natural_df["label"] == 1]

print(len(accepts), "/", len(natural_df))
print(natural_df["label"].value_counts(normalize=True))